[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C46_Graph_ML_Course/04_graph_transformer/04_graph_transformer.ipynb)

# 04 · 图 Transformer（用 numpy 从零）

目标：从零实现 **拉普拉斯位置编码 (LapPE)** 与 **全局多头自注意力**，搭一个玩具图 Transformer，验证它能看见 MPNN 看不见的全局结构、并直面**符号歧义**。

路线：LapPE(谱位置编码) → 符号歧义(必踩的坑+随机翻转) → 全局自注意力 → 多头 → RWSE 结构编码 → 玩具图 Transformer 层 → ✏️ 练习 → 📖 答案 → 🧪 过挤压：全局注意力 vs 逐跳传播胶囊。

> 心智模型：**图 Transformer = 把图当全连接 token 集做自注意力（一步全局），用 LapPE/结构编码把图拓扑重新注入；位置编码不是锦上添花，是它「看见图」的唯一途径。**

## 1 · 拉普拉斯位置编码 (LapPE)：图上的「正弦波」

取归一化拉普拉斯最小的 k 个**非平凡**特征向量（跳过 λ₁=0 的常向量）作为每个节点的 k 维「坐标」。
这是 NLP 正弦位置编码在图上的推广（拉普拉斯特征向量 = 图的傅里叶基）。

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def sym_norm_laplacian(A):
    deg = np.maximum(A.sum(1), 1e-12)
    dinv_sqrt = np.diag(1.0/np.sqrt(deg))
    return np.eye(len(A)) - dinv_sqrt @ A @ dinv_sqrt

def laplacian_pe(A, k):
    '''取 L_sym 最小的 k 个非平凡特征向量(跳过第0个常向量)作位置编码。返回 (n,k)。'''
    L = sym_norm_laplacian(A)
    w, U = np.linalg.eigh(L)              # 升序
    return U[:, 1:k+1]                    # 跳过 λ1=0 对应的常向量

# 一个有两个社区的小图
edges = [(0,1),(1,2),(0,2),(3,4),(4,5),(3,5),(2,3)]
n = 6; A = np.zeros((n,n))
for i,j in edges: A[i,j]=A[j,i]=1.0
pe = laplacian_pe(A, k=2)
print('LapPE (每行=一个节点的位置坐标):\n', pe)
assert pe.shape == (n, 2)
# 第一个非平凡特征向量(Fiedler)应区分两个社区(符号)
comm = (pe[:,0] > 0).astype(int)
assert len(set(comm[:3].tolist())) == 1 and len(set(comm[3:].tolist())) == 1, \
    'Fiedler 坐标应把两个三角形社区分到两侧'
print('✅ LapPE 的第一维(Fiedler)把两个社区分到了正负两侧——位置编码看见了全局结构')

**LapPE 区分对称节点**：在一条路径图上，两端的节点结构对称，但 LapPE 给它们不同的坐标（位置不同）。

In [ ]:
# 路径图 0-1-2-3-4
n2 = 5; Ap = np.zeros((n2,n2))
for i in range(n2-1): Ap[i,i+1]=Ap[i+1,i]=1.0
pe_path = laplacian_pe(Ap, k=1)
print('路径图 LapPE(第一维):', pe_path[:,0])
# 沿路径单调变化(低频模式)，给每个位置不同坐标
assert pe_path[0,0] * pe_path[-1,0] < 0, '路径两端应在 Fiedler 向量的相反侧'
print('✅ LapPE 沿路径给出渐变坐标：注意力据此能分辨「谁在路径的哪一端」')

## 2 · 符号歧义：LapPE 必踩的坑

特征向量 u 与 −u 都是合法特征向量，求解器返回哪个是**任意的**。同一张图的 LapPE 符号可能随机翻转——
这会让模型把同一节点当成两个不同位置。**对策：训练时随机翻转每列符号（数据增强）**。

In [ ]:
# 演示符号歧义：同一个 L，u 和 -u 都是特征向量
L = sym_norm_laplacian(A)
w, U = np.linalg.eigh(L)
u2 = U[:, 1]
# 验证 -u2 也是同一特征值的特征向量
assert np.allclose(L @ u2, w[1] * u2)
assert np.allclose(L @ (-u2), w[1] * (-u2))
print('u 和 -u 都满足 Lu=λu：符号是任意的 ⚠️')

def random_sign_flip(pe, seed=0):
    '''训练时增强：随机翻转每一列(每个特征向量)的符号。'''
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1.0, 1.0], size=pe.shape[1])
    return pe * signs

pe = laplacian_pe(A, k=2)
pe_flipped = random_sign_flip(pe, seed=3)
print('原始 PE 第1维:', pe[:,0])
print('翻转 PE 第1维:', pe_flipped[:,0])
# 翻转后仍是合法位置编码(只是符号变了)
assert np.allclose(np.abs(pe), np.abs(pe_flipped)), '翻转只改符号不改大小'
print('✅ 随机符号翻转增强：逼模型对符号鲁棒(用了 LapPE 就必须做这个！)')

## 3 · 全局自注意力：每个节点关注所有节点

标准自注意力 `softmax(QKᵀ/√d) V`，但作用在**全部节点**上（全连接，不只邻居）。
验证：注意力矩阵每行和=1、是全连接（每个节点关注所有节点，与边无关）。

In [ ]:
def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    e = np.exp(S); return e / e.sum(1, keepdims=True)

def self_attention(H, Wq, Wk, Wv):
    '''全局自注意力(单头)。返回 (输出, 注意力矩阵)。'''
    Q = H @ Wq; Kk = H @ Wk; V = H @ Wv
    dk = Q.shape[1]
    S = Q @ Kk.T / np.sqrt(dk)
    Aw = softmax_rows(S)
    return Aw @ V, Aw

rng = np.random.default_rng(0)
d = 4
H = rng.standard_normal((n, d))
Wq = rng.standard_normal((d,d))*0.5; Wk = rng.standard_normal((d,d))*0.5; Wv = rng.standard_normal((d,d))*0.5
out, Aw = self_attention(H, Wq, Wk, Wv)
print('注意力矩阵形状:', Aw.shape, '(全连接 n×n)')
assert np.allclose(Aw.sum(1), 1.0), '每行注意力和为1'
assert np.all(Aw > 0), '全连接：每个节点对所有节点都有非零注意力(与边无关)'
print('✅ 全局自注意力：每行和=1、全连接(一步建立任意距离依赖)')

**关键验证：纯注意力「看不见」图结构**。两张边结构不同但节点特征相同的图，纯自注意力输出**完全一样**——这正是必须加位置编码的原因。

In [ ]:
# 图1: 0-1-2-3-4-5 路径; 图2: 完全不同的边. 节点特征相同 H.
A1 = np.zeros((n,n))
for i in range(n-1): A1[i,i+1]=A1[i+1,i]=1.0
A2 = np.zeros((n,n))
for i,j in [(0,3),(1,4),(2,5),(0,5)]: A2[i,j]=A2[j,i]=1.0
out1, _ = self_attention(H, Wq, Wk, Wv)   # 注意：self_attention 根本没用到 A！
out2, _ = self_attention(H, Wq, Wk, Wv)
assert np.allclose(out1, out2), '纯自注意力不看边 -> 不同图同特征给相同输出'
print('⚠️ 纯自注意力对两张不同的图给出完全相同的输出——它看不见边！')
print('✅ 这证明：位置/结构编码是图 Transformer 看见图的唯一途径')

## 4 · 多头全局注意力

和 GAT/NLP 一样：K 个头各有独立 Q/K/V 投影，并行算后拼接再投影回去。

In [ ]:
def multihead_self_attention(H, heads, Wo):
    '''heads: list of (Wq,Wk,Wv). 各头输出拼接后过 Wo.'''
    outs = [self_attention(H, Wq, Wk, Wv)[0] for (Wq,Wk,Wv) in heads]
    cat = np.concatenate(outs, axis=1)
    return cat @ Wo

K = 3; dh = 4
heads = []
for h in range(K):
    r = np.random.default_rng(20+h)
    heads.append((r.standard_normal((d,dh))*0.5, r.standard_normal((d,dh))*0.5, r.standard_normal((d,dh))*0.5))
Wo = rng.standard_normal((K*dh, d))*0.5
out_mh = multihead_self_attention(H, heads, Wo)
print('多头输出形状:', out_mh.shape)
assert out_mh.shape == (n, d), '拼接 K 头(各 dh) -> Wo 投影回 d'
print('✅ 多头全局注意力：K 头拼接后投影回 d 维')

## 5 · 随机游走结构编码 (RWSE)

另一类位置/结构编码：随机游走 k 步**回到自身**的概率 `[P¹_vv, P²_vv, ..., Pᵏ_vv]`。
它编码节点的局部结构（在不在三角形/环里），与全局的 LapPE 互补。

In [ ]:
def rwse(A, k):
    '''随机游走结构编码：每个节点 k 步回到自身的概率。返回 (n,k).'''
    deg = np.maximum(A.sum(1), 1e-12)
    P = np.diag(1.0/deg) @ A          # 转移矩阵
    out = np.zeros((len(A), k))
    Pk = np.eye(len(A))
    for step in range(k):
        Pk = Pk @ P
        out[:, step] = np.diag(Pk)   # 回到自身的概率
    return out

# 三角形里的节点 vs 路径上的节点，RWSE 不同
se = rwse(A, k=4)
print('RWSE (每行=节点的回归概率指纹):\n', se)
# 三角形节点(0,1,2)有奇环 -> 2步即可回到自身概率高
assert se[0,1] > 0, '三角形节点2步能回到自身'
assert se.shape == (n, 4)
print('✅ RWSE 编码局部结构：三角形里的节点有独特的回归概率指纹')

## 6 · 玩具图 Transformer 层：LapPE + 全局注意力 + 残差/FFN

把零件拼成一层：① 注入 LapPE；② 全局多头注意力 + 残差;③ FFN + 残差。
**验证核心**：加了 LapPE 后，模型对两张不同图给出**不同**输出（对比 worked 3 的纯注意力）。

In [ ]:
def layernorm(x, eps=1e-5):
    mu = x.mean(1, keepdims=True); var = x.var(1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def graph_transformer_layer(A, H, k_pe=2, seed=0):
    '''最简图 Transformer 层：H + LapPE -> 全局注意力 -> 残差+FFN。'''
    rng = np.random.default_rng(seed)
    nn, d = H.shape
    pe = laplacian_pe(A, k=k_pe)
    Wpe = rng.standard_normal((k_pe, d))*0.5
    H0 = H + pe @ Wpe                      # 注入位置编码
    Wq=rng.standard_normal((d,d))*0.5; Wk=rng.standard_normal((d,d))*0.5; Wv=rng.standard_normal((d,d))*0.5
    attn_out, _ = self_attention(H0, Wq, Wk, Wv)
    H1 = layernorm(H0 + attn_out)          # 残差 + norm
    W1=rng.standard_normal((d,d))*0.5; W2=rng.standard_normal((d,d))*0.5
    ffn = np.maximum(H1 @ W1, 0) @ W2
    return layernorm(H1 + ffn)             # 残差 + norm

# 用同样的节点特征、两张不同的图
Hin = rng.standard_normal((n, 4))
outA = graph_transformer_layer(A1, Hin, seed=1)    # 路径图
outB = graph_transformer_layer(A2, Hin, seed=1)    # 另一个图
print('加 LapPE 后，两张不同图的输出差异:', np.abs(outA-outB).max())
assert not np.allclose(outA, outB), '加 LapPE 后应能区分不同图(对比 worked 3 纯注意力)'
print('✅ 玩具图 Transformer：LapPE 让它「看见」了图结构——不同图给不同输出！')

---
## ✏️ 练习 1：注意力距离偏置（Graphormer 风格）

实现 `attention_with_distance_bias(H, Wq, Wk, Wv, dist, bias_per_dist)`：在注意力分数上**加距离偏置** `QKᵀ/√d + bias[dist_ij]`，
让图上更近的节点更易互相关注。`dist` 是最短路径距离矩阵，`bias_per_dist[d]` 是距离 d 的可学偏置。

In [ ]:
def attention_with_distance_bias(H, Wq, Wk, Wv, dist, bias_per_dist):
    # TODO: Q=H@Wq; K=H@Wk; V=H@Wv; S = Q@K.T/sqrt(dk)
    #       加偏置：S[i,j] += bias_per_dist[dist[i,j]]（用花式索引 bias_per_dist[dist]）
    #       softmax 后乘 V，返回 (输出, 注意力矩阵)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
dist = np.array([[0,1,2],[1,0,1],[2,1,0]])
Hs = np.random.default_rng(0).standard_normal((3, 4))
Wq=Wk=Wv=np.eye(4)
# 偏置：距离越远偏置越负(越不关注)
bias = np.array([0.0, -1.0, -5.0])
out_b, Aw_b = attention_with_distance_bias(Hs, Wq, Wk, Wv, dist, bias)
assert np.allclose(Aw_b.sum(1), 1.0), '每行和为1'
# 节点0对远处节点2(dist=2,大负偏置)的注意力 应 < 对近处节点1(dist=1)
assert Aw_b[0,2] < Aw_b[0,1], '距离偏置应让远处节点获得更少注意力'
print('✅ 练习 1 通过：距离偏置让图上更近的节点更易互相关注(Graphormer 核心)')

## ✏️ 练习 2：符号不变的 LapPE 读出

为消除符号歧义，一个简单技巧：用对符号**不变**的量。实现 `sign_invariant_pe(pe)`，
返回 `|pe|`（逐元素绝对值）——它对 `pe -> -pe` 不变。验证翻转符号后结果一致。

In [ ]:
def sign_invariant_pe(pe):
    # TODO: 返回对列符号翻转不变的编码。最简单：逐元素绝对值 abs(pe)
    #       (注：这会丢失一些信息，SignNet 用 φ(u)+φ(-u) 更好；这里练直觉)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pe = laplacian_pe(A, k=3)
pe_flip = pe * np.array([-1.0, 1.0, -1.0])    # 翻转第1,3列
inv1 = sign_invariant_pe(pe)
inv2 = sign_invariant_pe(pe_flip)
assert np.allclose(inv1, inv2), '符号不变编码应对符号翻转一致'
print('✅ 练习 2 通过：|pe| 对符号翻转不变(SignNet 思想的最简版)')

## ✏️ 练习 3：最短路径距离矩阵（BFS）

距离编码需要最短路径。实现 `shortest_path_distances(A)`：用 BFS 算所有节点对的最短路径距离（无权图）。
不连通则距离为 `np.inf`。验证：对称、对角为0、相邻节点距离1。

In [ ]:
def shortest_path_distances(A):
    # TODO: 对每个源节点 s 做 BFS，记录到所有节点的跳数。
    #       返回 (n,n) 距离矩阵；不可达=inf；A[i,j]>0 表示边。
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Apath = np.zeros((4,4))
for i in range(3): Apath[i,i+1]=Apath[i+1,i]=1.0   # 0-1-2-3
D = shortest_path_distances(Apath)
assert np.allclose(np.diag(D), 0), '对角为0'
assert np.allclose(D, D.T), '距离对称'
assert D[0,1]==1 and D[0,3]==3, '相邻=1, 两端=3'
print('最短路径距离矩阵:\n', D)
print('✅ 练习 3 通过：BFS 最短路径距离正确(距离编码的输入)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def attention_with_distance_bias(H, Wq, Wk, Wv, dist, bias_per_dist):
    Q = H @ Wq; Kk = H @ Wk; V = H @ Wv
    dk = Q.shape[1]
    S = Q @ Kk.T / np.sqrt(dk)
    S = S + bias_per_dist[dist]            # 花式索引：每对加其距离的偏置
    Aw = softmax_rows(S)
    return Aw @ V, Aw

In [ ]:
# 练习 2 参考答案
def sign_invariant_pe(pe):
    return np.abs(pe)

In [ ]:
# 练习 3 参考答案
from collections import deque
def shortest_path_distances(A):
    nn = len(A)
    D = np.full((nn, nn), np.inf)
    for s in range(nn):
        D[s, s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for v in np.nonzero(A[u])[0]:
                if D[s, v] == np.inf:
                    D[s, v] = D[s, u] + 1
                    q.append(v)
    return D

---
## 🧪 真实数据胶囊：过挤压——全局注意力 vs 逐跳传播

构造一个有**瓶颈**的图（两个团 + 一条桥，即模块01的小谱隙结构）：信息从一端传到另一端必须挤过那条桥。
对比：① MPNN 逐跳传播信息穿过瓶颈的衰减；② 图 Transformer 全局注意力一步直达。这就是过挤压与图 Transformer 的动机。

In [ ]:
def barbell_graph(clique_size=5):
    '''哑铃图：两个团 + 一条桥(瓶颈)。'''
    nn = 2*clique_size
    A = np.zeros((nn,nn))
    for i in range(clique_size):           # 左团全连接
        for j in range(i+1, clique_size):
            A[i,j]=A[j,i]=1.0
    for i in range(clique_size, nn):       # 右团全连接
        for j in range(i+1, nn):
            A[i,j]=A[j,i]=1.0
    A[clique_size-1, clique_size]=A[clique_size, clique_size-1]=1.0   # 桥
    return A

A_bar = barbell_graph(5)
nn = len(A_bar)
# 谱隙小 = 有瓶颈
L = sym_norm_laplacian(A_bar)
lam2 = np.sort(np.linalg.eigvalsh(L))[1]
print(f'哑铃图 {nn} 节点，λ₂(谱隙)={lam2:.4f} (小 = 有瓶颈 = 易过挤压)')
assert lam2 < 0.5, '哑铃图应有小谱隙(瓶颈)'
print('✅ 哑铃图就绪：两团一桥，信息穿桥时会被挤压')

**🧪 胶囊练习**：在节点0(左团)放一个信号，用 GCN 式逐跳传播，看信号传到右团(节点9)有多弱（过挤压）。
实现 `signal_reaching_other_end(A, k)`：在节点0放单位信号，传播 k 步（行归一化传播），返回最远节点收到的信号量。补全传播那一行。

In [ ]:
def signal_reaching_other_end(A, k):
    nn = len(A)
    deg = np.maximum(A.sum(1), 1e-12)
    P = np.diag(1.0/deg) @ A          # 行归一化传播(随机游走)
    x = np.zeros(nn); x[0] = 1.0      # 信号放在节点0
    for _ in range(k):
        x = None                     # TODO: x @ P —— 传播一步
    return x[nn-1]                    # 最远节点(右团)收到多少

raise NotImplementedError  # 删除并补全传播

In [ ]:
# 自测：信号穿过瓶颈后严重衰减(过挤压)，而全局注意力一步就能直连两端
reach3 = signal_reaching_other_end(A_bar, 3)
print(f'3 步传播后，对端节点收到的信号 = {reach3:.4f}')
assert 0 < reach3 < 0.2, '信号穿过瓶颈后应严重衰减(过挤压)'
# 对比：图 Transformer 全局注意力中，节点0对节点9 一步就有直接(非零)连接
_, Aw = self_attention(np.random.default_rng(0).standard_normal((nn,4)),
                       np.eye(4), np.eye(4), np.eye(4))
assert Aw[0, nn-1] > 0, '全局注意力中两端一步直连(非零注意力)'
print(f'全局注意力中 节点0->节点9 的直接注意力 = {Aw[0,nn-1]:.4f} (一步直达，无衰减)')
print('✅ 胶囊练习通过：MPNN 信号穿瓶颈严重衰减(过挤压)；图 Transformer 一步直连绕开它')

In [ ]:
# 📖 胶囊参考答案
def signal_reaching_other_end(A, k):
    nn = len(A)
    deg = np.maximum(A.sum(1), 1e-12)
    P = np.diag(1.0/deg) @ A
    x = np.zeros(nn); x[0] = 1.0
    for _ in range(k):
        x = x @ P                    # 传播一步
    return x[nn-1]

reach3 = signal_reaching_other_end(A_bar, 3)
print(f'3 步传播后对端收到信号 = {reach3:.4f} (穿瓶颈衰减)')
assert 0 < reach3 < 0.2
print('✅ MPNN 受困于瓶颈(过挤压)，图 Transformer 全局注意力一步直达——这就是它的价值')

### 小结
- **图 Transformer = 全连接 token 自注意力(一步全局)**，绕开 MPNN 的局部性与**过挤压**。
- **纯注意力看不见边**：位置/结构编码是它「看见图」的唯一途径(worked 3 已证)。
- **LapPE = 拉普拉斯特征向量当位置编码**(图上的正弦波)；**必须处理符号歧义**(随机翻转)。
- **结构/距离编码**(度/RWSE/最短路径偏置)是另一条注入拓扑的路；**混合层**(局部+全局)是事实标准。

下一站：**模块 05 · 可扩展与应用** —— 采样/分块、过平滑/过挤压诊断，把前四块落到链接预测与图级分类。